# Install Libraries

In [68]:
!pip install fastapi nest-asyncio accelerate scipy torch pillow -q
!pip install transformers==4.41.0 -q
!pip install pyvips -q
!apt-get install -y libvips -q
!pip install uvicorn==0.29.0 -q

# Model xử lý tts (edge-tts của rany2 từ github)
!pip install edge-tts -q
!pip install nest_asyncio -q

# LLM groq
!pip install groq -q

Reading package lists...
Building dependency tree...
Reading state information...
libvips42 is already the newest version (8.12.1-1build1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


# Build class model (moondream2)

In [69]:
import torch
from transformers import AutoModelForCausalLM
from google.colab import userdata
from huggingface_hub import login

class Moondream2:
    def __init__(self):
        login(userdata.get('HF_TOKEN'))
        self.model = AutoModelForCausalLM.from_pretrained(
            "vikhyatk/moondream2",
            revision="2025-01-09",
            trust_remote_code=True,
            device_map={"": "cuda"}
        )

    def __call__(self, image, question):
        result = self.model.query(
            image=image,
            question=question
        )
        return result["answer"]

# LLM Class

In [70]:
from groq import Groq
from google.colab import userdata

class FoodStoryGenerator:
    def __init__(self):
        self.client = Groq(api_key=userdata.get('GROQ_API_KEY'))

    def __call__(self, food_desc):
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": f"""
                Dựa vào mô tả món ăn: "{food_desc}"
                Viết đoạn văn tiếng Việt hấp dẫn về:
                1. Lịch sử món ăn
                2. Đặc điểm nổi bật
                3. Cách thưởng thức
                """
            }]
        )
        return response.choices[0].message.content

# Build class model (edge-tts)

In [71]:
import edge_tts
import asyncio
import nest_asyncio
from IPython.display import Audio, display

nest_asyncio.apply()

class VietnameseTTS:
    def __init__(self, voice="vi-VN-HoaiMyNeural"):
        self.voice = voice

    async def _generate(self, text, output_path):
        communicate = edge_tts.Communicate(text=text, voice=self.voice)
        await communicate.save(output_path)

    def generate(self, text, output_path="/content/output.mp3"):
        asyncio.get_event_loop().run_until_complete(
            self._generate(text, output_path)
        )
        return output_path

    def __call__(self, text, output_path="/content/output.mp3"):
        audio_path = self.generate(text, output_path)
        display(Audio(audio_path, autoplay=True))
        return audio_path


# Initialize API and Class Model

In [72]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# Khởi tạo model
moondream  = Moondream2()
generator  = FoodStoryGenerator()
tts        = VietnameseTTS()

##GET / — Giới thiệu hệ thống

In [73]:
# GET / — Giới thiệu hệ thống
@app.get('/')
async def root():
    return {
        "name": "Food Story API",
        "description": "Đây là hệ thống API để người dùng có thể upload những hình ảnh về ẩm thực từ đó hệ thống sẽ trả về dạng text và đọc nội dung của văn bản",
        "technologies": "Upload ảnh món ăn → nhận diện (vikhyatk/moondream2) → generate lịch sử tiếng Việt (groq) → TTS (edge-tts của rany2)",
        "endpoints": {
            "GET  /":        "Giới thiệu hệ thống",
            "GET  /health":  "Kiểm tra trạng thái",
            "POST /predict": "Nhận diện món ăn từ ảnh",
            "POST /generate":"Nhận diện + generate story + TTS",
            "GET  /audio":   "Trả về file audio"
        }
    }

## GET /health — Kiểm tra trạng thái

In [74]:
# GET /health — Kiểm tra trạng thái
@app.get('/health')
async def health():
    status = {}

    # Kiểm tra Moondream
    try:
        moondream.model
        status["moondream"] = "loaded"
    except:
        status["moondream"] = "error"

    # Kiểm tra Groq
    try:
        generator.client.models.list()
        status["groq"] = "loaded"
    except:
        status["groq"] = "error"

    # Kiểm tra TTS
    try:
        tts.voice
        status["tts"] = "loaded"
    except:
        status["tts"] = "error"

    # Kiểm tra GPU
    import torch
    status["gpu"] = f"{torch.cuda.get_device_name(0)} is ok" if torch.cuda.is_available() else "CPU ⚠️"

    return {
        "status": "OK" if all("loaded" in v for v in status.values()) else "degraded",
        "models": status
    }

## POST /predict — Nhận diện món ăn

In [75]:
# POST /predict — Nhận diện món ăn
from fastapi import UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from PIL import Image, UnidentifiedImageError
import io

# Chỉnh lại các giới hạn
MAX_FILE_SIZE = 10 * 1024 * 1024  # 10MB
ALLOWED_TYPES = ["image/jpeg", "image/png", "image/webp", "image/jpg"]


@app.post('/predict')
async def predict(file: UploadFile = File(...)):

    # 1. Kiểm tra content type
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(
            status_code=400,
            detail={
                "success": False,
                "message": f"Chỉ chấp nhận file ảnh: {', '.join(ALLOWED_TYPES)}. Nhận được: {file.content_type}"
            }
        )

    # 2. Đọc & kiểm tra kích thước
    contents = await file.read()

    if len(contents) > MAX_FILE_SIZE:
        raise HTTPException(
            status_code=400,
            detail={"success": False, "message": "File quá lớn. Tối đa 10MB."}
        )

    if len(contents) == 0:
        raise HTTPException(
            status_code=400,
            detail={"success": False, "message": "File rỗng."}
        )

    # 3. Kiểm tra file có phải ảnh hợp lệ không
    try:
        image = Image.open(io.BytesIO(contents))
        image.verify()
        image = Image.open(io.BytesIO(contents))
    except (UnidentifiedImageError, Exception):
        raise HTTPException(
            status_code=400,
            detail={"success": False, "message": "File không phải ảnh hợp lệ hoặc bị hỏng."}
        )

    # 4. Đọc ảnh
    image = Image.open(io.BytesIO(contents))

    # 5. Nhận diện
    try:
        food_desc = moondream(
            image=image,
            question="What food is this? Describe it briefly."
        )
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail={"success": False, "message": f"Lỗi nhận diện ảnh: {str(e)}"}
        )

    return {
        "food_description": food_desc
    }


## POST /generate — Full pipeline: nhận diện + story + TTS

In [76]:
# POST /generate — Full pipeline: nhận diện + story + TTS
@app.post('/generate')
async def generate(file: UploadFile = File(...)):

    # === VALIDATE ẢNH ===

    # Check type
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(
            status_code=400,
            detail={"success": False, "message": f"Chỉ chấp nhận: {', '.join(ALLOWED_TYPES)}"}
        )

    contents = await file.read()

    # Check size
    if len(contents) == 0:
        raise HTTPException(status_code=400, detail={"success": False, "message": "File rỗng."})
    if len(contents) > MAX_FILE_SIZE:
        raise HTTPException(status_code=400, detail={"success": False, "message": "File quá lớn. Tối đa 10MB."})

    # Check valid
    try:
        image = Image.open(io.BytesIO(contents))
        image.verify()
        image = Image.open(io.BytesIO(contents))
    except Exception:
        raise HTTPException(status_code=400, detail={"success": False, "message": "File ảnh không hợp lệ."})

    # === Nhận diện ===
    try:
        food_desc = moondream(image=image, question="What food is this? Describe it briefly.")
    except Exception as e:
        raise HTTPException(status_code=500, detail={"success": False, "message": f"Lỗi nhận diện ảnh: {str(e)}"})

    if not food_desc or not food_desc.strip():
        raise HTTPException(status_code=500, detail={"success": False, "message": "Không nhận diện được món ăn."})

    # === Generate story ===
    try:
        story = generator(food_desc)
    except Exception as e:
        raise HTTPException(status_code=500, detail={"success": False, "message": f"Lỗi tạo story: {str(e)}"})

    if not story or not story.strip():
        raise HTTPException(status_code=500, detail={"success": False, "message": "Không tạo được story."})

    # === TTS ===
    try:
        audio_path = tts.generate(story)
    except Exception as e:
        raise HTTPException(status_code=500, detail={"success": False, "message": f"Lỗi tạo audio: {str(e)}"})

    # === RESPONSE ===
    return {
        "success": True,
        "data": {
            "food_description": food_desc,
            "story": story,
            "audio_url": "/audio"
        }
    }

## GET /audio — Trả về file audio

In [77]:
# GET /audio — Trả về file audio
@app.get('/audio')
async def get_audio():
    audio_path = "/content/output.mp3"

    # --- Kiểm tra file audio có tồn tại không ---
    if not os.path.exists(audio_path):
        raise HTTPException(
            status_code=404,
            detail={"success": False, "message": "Chưa có file audio. Hãy gọi /generate trước."}
        )

    return FileResponse(
        audio_path,
        media_type="audio/mpeg",
        filename="food_story.mp3"
    )

## Running in localhost

In [78]:
import threading
import uvicorn
import asyncio

def run_server():
    asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(3)


INFO:     Started server process [9796]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


# Call Local API (Testing)

In [82]:
import requests
from google.colab import files
from IPython.display import Audio, display
import io
import os

BASE_URL = "http://127.0.0.1:8000"

# 1. Test GET /
print("--- 1. Testing / ---")
try:
    root_resp = requests.get(f"{BASE_URL}/")
    print(root_resp.json())
except Exception as e:
    print(f"Lỗi kết nối server: {e}")

# 2. Test GET /health
print("\n--- 2. Testing /health ---")
try:
    health_resp = requests.get(f"{BASE_URL}/health")
    print(health_resp.json())
except Exception as e:
    print(f"Lỗi: {e}")

# 3. Upload ảnh
print("\n--- Upload ảnh ---")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    file_content = uploaded[filename]
    print(f"Đã upload: {filename}")

    # 4. Test POST /predict
    print("\n--- 3. Testing /predict ---")
    try:
        predict_resp = requests.post(
            f"{BASE_URL}/predict",
            files={'file': (filename, file_content, 'image/jpeg')}
        )
        print(f"Status: {predict_resp.status_code}")
        print(predict_resp.json())
    except Exception as e:
        print(f"Lỗi: {e}")

    # 5. Test POST /generate
    print("\n--- 4. Testing /generate ---")
    try:
      generate_resp = requests.post(
          f"{BASE_URL}/generate",
          files={'file': (filename, file_content, 'image/jpeg')}
      )
      print(f"Status: {generate_resp.status_code}")
      result = generate_resp.json()
      data = result['data']
      print(f"Description: {data['food_description']}")
      print(f"Story: {data['story'][:150]}...")
    except Exception as e:
      print(f"Lỗi: {e}")

    # 6. Test GET /audio
    print("\n--- 5. Testing /audio ---")
    try:
        audio_resp = requests.get(f"{BASE_URL}/audio")
        print(f"Status: {audio_resp.status_code}")
        if audio_resp.status_code == 200:
            with open("test_output.mp3", "wb") as f:
                f.write(audio_resp.content)
            print("Lưu audio thành công!")
            display(Audio("test_output.mp3", autoplay=True))
        else:
            print(f"Lỗi: {audio_resp.status_code}")
    except Exception as e:
        print(f"Lỗi: {e}")

else:
    print("Chưa upload ảnh!")

--- 1. Testing / ---
INFO:     127.0.0.1:58776 - "GET / HTTP/1.1" 200 OK
{'name': 'Food Story API', 'description': 'Đây là hệ thống API để người dùng có thể upload những hình ảnh về ẩm thực từ đó hệ thống sẽ trả về dạng text và đọc nội dung của văn bản', 'technologies': 'Upload ảnh món ăn → nhận diện (vikhyatk/moondream2) → generate lịch sử tiếng Việt (groq) → TTS (edge-tts của rany2)', 'endpoints': {'GET  /': 'Giới thiệu hệ thống', 'GET  /health': 'Kiểm tra trạng thái', 'POST /predict': 'Nhận diện món ăn từ ảnh', 'POST /generate': 'Nhận diện + generate story + TTS', 'GET  /audio': 'Trả về file audio'}}

--- 2. Testing /health ---
INFO:     127.0.0.1:58778 - "GET /health HTTP/1.1" 200 OK
{'status': 'degraded', 'models': {'moondream': 'loaded', 'groq': 'loaded', 'tts': 'loaded', 'gpu': 'Tesla T4 is ok'}}

--- Upload ảnh ---


Saving pho.jpg to pho (14).jpg
Đã upload: pho (14).jpg

--- 3. Testing /predict ---
INFO:     127.0.0.1:56410 - "POST /predict HTTP/1.1" 200 OK
Status: 200
{'food_description': ' This is a bowl of beef pho soup.'}

--- 4. Testing /generate ---
INFO:     127.0.0.1:56416 - "POST /generate HTTP/1.1" 200 OK
Status: 200
Description:  This is a bowl of beef pho soup.
Story: Phở bò, một trong những món ăn quốc hồn quốc túy của Việt Nam, đã có một lịch sử lâu đời và hấp dẫn. Từ những ngày đầu tiên xuất hiện ở miền Bắc, đặc ...

--- 5. Testing /audio ---
INFO:     127.0.0.1:49580 - "GET /audio HTTP/1.1" 200 OK
Status: 200
Lưu audio thành công!


# Creating Tunnel

In [84]:
import subprocess, time, re

with open("/content/tunnel.log", "w") as log:
    subprocess.Popen(
        ["ssh", "-p", "443",
         "-o", "StrictHostKeyChecking=no",
         "-R0:localhost:8000",
         "qr@a.pinggy.io"],
        stdout=log, stderr=log
    )

time.sleep(8)

with open("/content/tunnel.log", "r") as f:
    content = f.read()

urls = re.findall(r'https?://[^\s]+pinggy-free\.link', content)
if urls:
    public_url = urls[0]
    print(f"URL public: {public_url}")
else:
    print("Chưa thấy URL, chạy lại cell này!")

URL public: http://uvakn-34-125-243-6.run.pinggy-free.link


# Testing public API

In [85]:
import requests
from google.colab import files
from IPython.display import Audio, display
import io

BASE_PUBLIC_URL = "http://uvakn-34-125-243-6.run.pinggy-free.link"

In [86]:
# 1. Test GET /
print("--- 1. Testing / ---")
try:
    root_resp = requests.get(f"{BASE_PUBLIC_URL}/")
    print(root_resp.json())
except Exception as e:
    print(f"Lỗi kết nối server: {e}")

--- 1. Testing / ---
INFO:     34.125.243.6:0 - "GET / HTTP/1.1" 200 OK
{'name': 'Food Story API', 'description': 'Đây là hệ thống API để người dùng có thể upload những hình ảnh về ẩm thực từ đó hệ thống sẽ trả về dạng text và đọc nội dung của văn bản', 'technologies': 'Upload ảnh món ăn → nhận diện (vikhyatk/moondream2) → generate lịch sử tiếng Việt (groq) → TTS (edge-tts của rany2)', 'endpoints': {'GET  /': 'Giới thiệu hệ thống', 'GET  /health': 'Kiểm tra trạng thái', 'POST /predict': 'Nhận diện món ăn từ ảnh', 'POST /generate': 'Nhận diện + generate story + TTS', 'GET  /audio': 'Trả về file audio'}}


In [87]:
# 2. Test GET /health
print("\n--- 2. Testing /health ---")
try:
    health_resp = requests.get(f"{BASE_PUBLIC_URL}/health")
    print(health_resp.json())
except Exception as e:
    print(f"Lỗi: {e}")


--- 2. Testing /health ---
INFO:     34.125.243.6:0 - "GET /health HTTP/1.1" 200 OK
{'status': 'degraded', 'models': {'moondream': 'loaded', 'groq': 'loaded', 'tts': 'loaded', 'gpu': 'Tesla T4 is ok'}}


In [88]:
# 3. Test POST /predict
print("\n--- Upload ảnh ---")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    file_content = uploaded[filename]
    print(f"Đã upload: {filename}")


print("\n--- 3. Testing /predict ---")

try:
  predict_resp = requests.post(
    f"{BASE_PUBLIC_URL}/predict",
    files={'file': (filename, file_content, 'image/jpeg')}
  )

  print(f"Status: {predict_resp.status_code}")
  print(predict_resp.json())

except Exception as e:
  print(f"Lỗi: {e}")


--- Upload ảnh ---


Saving hello.pdf to hello.pdf
Đã upload: hello.pdf

--- 3. Testing /predict ---
INFO:     34.125.243.6:0 - "POST /predict HTTP/1.1" 400 Bad Request
Status: 400
{'detail': {'success': False, 'message': 'File không phải ảnh hợp lệ hoặc bị hỏng.'}}


In [89]:
# 4. Test POST /generate

print("\n--- Upload ảnh ---")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    file_content = uploaded[filename]
    print(f"Đã upload: {filename}")


print("\n--- 4. Testing /generate ---")
try:
  generate_resp = requests.post(
    f"{BASE_PUBLIC_URL}/generate",
    files={'file': (filename, file_content, 'image/jpeg')}
  )
  print(f"Status: {generate_resp.status_code}")

  data = generate_resp.json()
  print(f"Description: {data['data']['food_description']}")
  print(f"Story: {data['data']['story'][:150]}...")


except Exception as e:
  print(f"Lỗi: {e}")


--- Upload ảnh ---


Saving Banh_chung.webp to Banh_chung (3).webp
Đã upload: Banh_chung (3).webp

--- 4. Testing /generate ---
INFO:     34.125.243.6:0 - "POST /generate HTTP/1.1" 200 OK
Status: 200
Description:  This is a type of wrapped food, possibly a dumpling or a wrapped noodle, that is wrapped in green leaves and decorated with red and white paper.
Story: Món ăn này là một loại thức ăn được bọc, có thể là bánh bao hoặc mì bọc, được bọc trong lá xanh và trang trí bằng giấy đỏ và trắng. Dưới đây là thông ...


In [90]:
# 5. Test GET /audio
print("\n--- 5. Testing /audio ---")
try:
    audio_resp = requests.get(f"{BASE_PUBLIC_URL}/audio")
    print(f"Status: {audio_resp.status_code}")
    if audio_resp.status_code == 200:
        with open("test_output.mp3", "wb") as f:
            f.write(audio_resp.content)
        print("Lưu audio thành công!")
        display(Audio("test_output.mp3", autoplay=True))
    else:
        print(f"Lỗi: {audio_resp.status_code}")
except Exception as e:
    print(f"Lỗi: {e}")


--- 5. Testing /audio ---
INFO:     34.125.243.6:0 - "GET /audio HTTP/1.1" 200 OK
Status: 200
Lưu audio thành công!
